In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import geopandas as gpd
import os

In [ ]:
filePath = "chile_ebd_relSep-2026.parquet"
df = pd.read_parquet(filePath)
print("Data Fully Loaded!")
display(df.head().T)

In [ ]:
rows, cols = df.shape
colNames = list(df.columns)
print("Rows:", rows)
print("Columns:", cols)
print("Column Names:\n\t-", "\n\t- ".join([f"{c}:\n\t\t- Type: {df[c].dtype}\n\t\t- Unique Values: {df[c].nunique()}\n\t\t- {f'Max Length: {df[c].astype(str).str.len().max()}' if str(df[c].dtype) == 'object' else f'Range: {df[c].min()}, {df[c].max()}'}" for c in colNames]))

In [ ]:
colNamesForMatrix = [ c for c in colNames if c not in ["GLOBAL UNIQUE IDENTIFIER", "LAST EDITED DATE", "OBSERVATION DATE", "TIME OBSERVATIONS STARTED"] ]
rsMatrix = pd.DataFrame(-1, index = colNamesForMatrix, columns = colNamesForMatrix)
for i, col1 in enumerate(colNamesForMatrix):
    if df[col1].nunique() <= 1:
        continue
    print(f"- {col1}:")
    for col2 in colNamesForMatrix[i + 1:]:
        if df[col2].nunique() <= 1:
            continue
        oneToN = df.groupby(col1)[col2].nunique().max() <= 1
        nToOne = df.groupby(col2)[col1].nunique().max() <= 1
        if oneToN and nToOne:
            values = ["1:1"] * 2
        elif oneToN:
            values = ["1:n", "n:1"]
        elif nToOne:
            values = ["n:1", "1:n"]
        else:
            values = ["n:n"] * 2
        if oneToN or nToOne:
            print(f"\t- {col1} - {col2}: {values[0]}")
        rsMatrix.loc[col1, col2] = values[0]
        rsMatrix.loc[col2, col1] = values[1]

In [ ]:
def parseDemographics(text):
    if pd.isna(text):
        return []
    pattern = r"([^,]+),\s*([^\(]+)\s*\((\d+)\)"
    result = []
    for item in text.split("; "):
        match = re.search(pattern, item.strip())
        if match:
            result.append(
                {
                    "SEX": match.group(1).strip(),
                    "AGE": match.group(2).strip(),
                    "QUANTITY": int(match.group(3)),
                }
            )
    return result

os.makedirs("cleaned_ebird_data", exist_ok = True)
chunkSize = 500000
for i, start in enumerate(range(0, len(df), chunkSize)):
    chunk = df.iloc[start:start + chunkSize].copy()
    chunk = chunk.drop(columns=["BCR CODE", "USFWS CODE", "ATLAS BLOCK", "OBSERVATION DATE", "REASON"], errors = "ignore")
    chunk["LAST EDITED DATE"] = pd.to_datetime(chunk["LAST EDITED DATE"])
    chunk["OBSERVATION COUNT"] = pd.to_numeric(chunk["OBSERVATION COUNT"], errors = "coerce")
    chunk["TIME OBSERVATIONS STARTED"] = pd.to_datetime(df["OBSERVATION DATE"].iloc[start:start + chunkSize].str.cat(chunk["TIME OBSERVATIONS STARTED"], sep = " "))
    chunk["AGE/SEX"] = chunk["AGE/SEX"].apply(parseDemographics)
    chunk.to_parquet(f"cleaned_ebird_data/chunk_{i}.parquet", index = False)
    print(f"Processed and saved chunk {i + 1}")
print("All chunks processed and saved to disk.")

In [ ]:
df1 = pd.read_parquet("cleaned_ebird_data/")
matchingIndexes = []
for idx, val in df1["AGE/SEX"].items():
    if isinstance(val, (list, np.ndarray)) and len(val) > 0:
        matchingIndexes.append(idx)
    if len(matchingIndexes) >= 5:
        break
display(df1.loc[matchingIndexes].T)

In [ ]:
print("Column Names:")
for c in list(df1.columns):
    if c == "AGE/SEX":
        continue
    isUnhashable = df1[c].apply(lambda x: isinstance(x, (list, dict, set))).any()
    workingSeries = df1[c].apply(lambda x: str(x) if isinstance(x, (list, dict, set)) else x) if isUnhashable else df1[c]
    dtype = df1[c].dtype
    uniqueCount = workingSeries.nunique()
    if str(dtype) == 'object':
        cleanSeries = workingSeries.dropna()
        maxLen = max((len(str(val)) for val in cleanSeries), default = 0)
        detail = f"Max Length: {maxLen}"
    else:
        detail = f"Range: {df1[c].min()}, {df1[c].max()}"
    print(f"\t- {c}:\n\t\t- Type: {dtype}\n\t\t- Unique Values: {uniqueCount}\n\t\t- {detail}")

[Visualize](https://mermaid.live/edit#pako:eNqtWG1v2zYQ_isCPy1AXNhS_RJ9y1pnC7KtRWAMRWFAoKWLrUYSNUrynDr57zu9UCFF0nOK-pOte-6Fd8e7Rz6SkEVAfAL8Y0y3nKZrvs4c_IS0hC3jT85RPKk_t3-tlr8t7504cj7fyYK_r-8__H59_8tkeuFkNAUhexFfihzCGIo3mpuOL5yQpSnLAtnqEFKg6ayMH-IwMDov6YFlqmuh7qJ6Iw5CloWQl4EWjIiygbEUnTAeATdBRNZqIzdGI10edIBynmojcOec_hVtSYSiceHAgZWIqEuvpWrDAaI42wY_2ABoXjNhdWIuiWICY7SUw5zr3kfIqqwchi-56OQmD30zj3uY3s8lujcbr9Ua8SnbwwhONENtyny8J3sArfz_TieFaQ7A601pEcQbanZf66Bw6LtXTFhIkxiDK59yeHt7KepW4_bM9Pr2ITbInz01_THNEHcs-TOOFDyCpX_ZpgC-B245CCZCIOwHeT-WUIyHcYRYiydaxjhnfqAkXu_k1YTmI-esZCFLrNdRAE7deD1bwzh6K8Y1gNJvEJbm83VCtN9OUOCFtT1aTw28dlTIqF9vVw7NUbwHPdUFTfOknmuwRy8nxoeCk2I60bFyX9_cndM0N3dnFcO8yEz5UpEfr1fL1e2fS6U7cOLwUkbd_PHpeuVEFW_laZxVpZrRFgEPD4yXQRSjBVzUwWNqxVAONNhRU9RZlW7qu9BlQa9ckgRimXLI0dprFYcJ33JW5dL5Zdxq-WXlhDsIHxMMuNniCCtO3T3L1agdJWxDk6DK4n8qOK8Z7A1km1QGGmQpZ0LxRLieMTVBJO0mOc9yyZtRetaitzbkBnZ0HzOucBZRsx0tghSt0KGAwz6Gf9UCNoWRyZWxLHT71ilYJ91MfOHwszg0RhW83dzJDjLf7dqRncHCwS5s75cSufZG8fw8Gj0f23bzZRo3fFdQga_MWaX1HUrqOF9rZTu77bSFwBhPT1dNrpQOlvunA3dF87ukym0xRLSZNY2GIdJWzyH57fQamucrhFPlsR2uJT2-RAsHdLPDiVXjyzRJ5oUaThAlMwfU4N2m10idOI8y3Xx59Wn0yaIi7UErD-pUxTbUwupJjcWHskaHJMSuM1yrFvJgakbr2CeXZMvjiPglr-CSpMBTWv8kx9r2mpQ7wIFDfPzKIaoOo4jyxxHGzviarLMX1M9p9pWxVJjA5bfdEf-BJgX-qvJ6E3T_IfQQyPA9-UPdIsSfeJPGBvGP5ED8q8W7-WIx99zZ1HPfT-fuJXki_sh1x_jcnXmet5jM5vPJ7OWSfG_cTt5duR4-ms2uZtPZfDpevPwH8Fo6JA)

```mermaid
erDiagram
    category {
        INTEGER id PK
        VARCHAR(15) name
    }
    species {
        INTEGER id PK
        VARCHAR(50) common_name
        VARCHAR(50) scientific_name
    }
    taxon {
        VARCHAR(20) taxon_concept_id PK
        INTEGER taxonomic_order
        INTEGER category_id FK
        INTEGER species_id FK
        VARCHAR(50) subspecies_common_name
        VARCHAR(50) subspecies_scientific_name
        VARCHAR(5) exotic_code
    }
    breeding_category {
        INTEGER id PK
        VARCHAR(5) breeding_category
    }
    breeding {
        VARCHAR(5) breeding_code PK
        INTEGER category_id FK
    }
    country {
        VARCHAR(5) country_code PK
        VARCHAR(10) country
    }
    state {
        VARCHAR(10) state_code PK
        VARCHAR(5) country_code FK
        VARCHAR(50) state
    }
    county {
        VARCHAR(10) county_code PK
        VARCHAR(10) state_code FK
        VARCHAR(30) county
    }
    iba {
        VARCHAR(30) iba_code PK
    }
    locality_type {
        INTEGER id PK
        VARCHAR(5) locality_type
    }
    locality {
        VARCHAR(10) locality_id PK
        VARCHAR(10) county_code FK
        VARCHAR(30) iba_code FK
        VARCHAR(200) locality
        INTEGER type_id FK
    }
    observer {
        VARCHAR(15) observer_id PK
        VARCHAR(40) observer_orcid_id
    }
    observation_type {
        INTEGER id PK
        VARCHAR(30) observation_type
    }
    protocol {
        VARCHAR(5) protocol_code PK
        INTEGER type_id FK
        VARCHAR(30) protocol_name
    }
    project {
        INTEGER project_identifiers PK
        VARCHAR(100) project_names
        BIT approved
    }
    sampling_event {
        VARCHAR(10) sampling_event_identifier PK
        VARCHAR(10) locality_id FK
        VARCHAR(15) observer_id FK
        VARCHAR(5) protocol_code FK
        INTEGER project_identifiers FK
        DATETIME observation_start
        FLOAT duration_minutes
        FLOAT effort_distance_km
        FLOAT effort_area_ha
        INTEGER number_observers
        BIT all_species_reported
        VARCHAR(10) group_identifier
        TEXT checklist_comments
    }
    observation {
        VARCHAR(50) global_unique_identifier PK
        VARCHAR(10) sampling_event_identifier FK
        VARCHAR(20) taxon_concept_id FK
        DATETIME last_edited_date
        INTEGER observation_count
        VARCHAR(5) breeding_code FK
        VARCHAR(5) behavior_code
        BIT has_media
        BIT reviewed
        TEXT species_comments
    }
    age {
        INTEGER id PK
        VARCHAR(10) name
    }
    sex {
        INTEGER id PK
        VARCHAR(15) name
    }
    age_sex {
        INTEGER id PK
        VARCHAR(50) global_unique_identifier FK
        INTEGER age_id FK
        INTEGER sex_id FK
        INTEGER number
    }

    category ||--|{ taxon: category_id
    species ||--|{ taxon: species_id
    taxon ||--|{ observation: taxon_concept_id
    breeding_category ||--|{ breeding: category_id
    breeding ||--|{ observation: breeding_code
    age ||--|{ age_sex: age_id
    sex ||--|{ age_sex: sex_id
    observation ||--|{ age_sex: global_unique_identifier
    country ||--|{ state: country_code
    state ||--|{ county: state_code
    county ||--|{ locality: county_code
    iba ||--|{ locality: iba_code
    locality_type ||--|{ locality: type_id
    locality ||--|{ sampling_event: locality_id
    observer ||--|{ sampling_event: observer_id
    observation_type ||--|{ protocol: type_id
    protocol ||--|{ sampling_event: protocol_code
    project ||--|{ sampling_event: project_identifiers
    sampling_event ||--|{ observation: sampling_event_identifier
```

In [7]:
gdf = gpd.GeoDataFrame(df1, geometry = gpd.points_from_xy(df1["LONGITUDE"], df1["LATITUDE"]), crs = "EPSG:4326")
gdf.to_file("chile_ebd_relSep-2026.shp", driver = "ESRI Shapefile")

C:\Users\lucia\AppData\Local\Temp\ipykernel_27472\3908764371.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file("chile_ebd_relSep-2026.shp", driver = "ESRI Shapefile")
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'GLOBAL UNIQUE IDENTIFIER' to 'GLOBAL UNI'
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'LAST EDITED DATE' to 'LAST EDITE'
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Field LAST EDITE create as date field, though DateTime requested.
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'TAXONOMIC ORDER' to 'TAXONOMIC'
  ogr_write(
C:\Users\lucia\anaconda3\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'TAXON 